In [43]:
import librosa
from music2latent import EncoderDecoder

encdec = EncoderDecoder()

In [44]:
sr = 44100
audio_path1 = '../wav24k/percussion/tabla.wav'
wv1, _ = librosa.load(audio_path1, sr=sr)
latent1 = encdec.encode(wv1)

audio_path2 = '../wav24k/percussion/amen_mono_24k.wav'
wv2, _ = librosa.load(audio_path2, sr=sr)
latent2 = encdec.encode(wv2)

/Users/sangarshananveera/dev/UPF/GenAI-Part2/latent-granular-resynthesis/.venv/lib/python3.13/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


In [45]:
import torch
import torch.nn.functional as F

def slerp(alpha, a, b, eps=1e-8):
    a_norm = F.normalize(a, dim=-1)
    b_norm = F.normalize(b, dim=-1)
    dot = (a_norm * b_norm).sum(dim=-1, keepdim=True).clamp(-1 + eps, 1 - eps)
    omega = dot.acos()
    sin_omega = omega.sin()
    parallel = sin_omega.abs() < eps
    coeff_a = torch.where(parallel, torch.tensor(1 - alpha), ((1 - alpha) * omega).sin() / sin_omega)
    coeff_b = torch.where(parallel, torch.tensor(alpha),     (      alpha  * omega).sin() / sin_omega)
    return coeff_a * a + coeff_b * b


def tokui_style_transfer(
    target_latents: torch.Tensor,
    pool_latents: torch.Tensor,
    alpha: float = 0.5,
    mode: str = "slerp",  # "lerp" | "slerp"
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    For each token in target, find the most similar token in pool
    (cosine similarity), then interpolate between them.

    music2latent shape: (1, D, T)
    alpha: 0.0 = pure target, 1.0 = pure pool
    """
    orig_shape = target_latents.shape  # (1, D, T_target)

    # (1, D, T) → (T, D) for both
    t_flat = target_latents.squeeze(0).T   # (T_target, D)
    p_flat = pool_latents.squeeze(0).T     # (T_pool,   D)

    # Cosine nearest-neighbour lookup
    t_norm = F.normalize(t_flat, dim=-1)
    p_norm = F.normalize(p_flat, dim=-1)
    sim = torch.mm(t_norm, p_norm.T)      # (T_target, T_pool)
    match_indices = sim.argmax(dim=-1)    # (T_target,)
    matched = p_flat[match_indices]       # (T_target, D)

    # Interpolate each target token with its matched pool token
    if mode == "lerp":
        hybrid = torch.lerp(t_flat, matched, alpha)
    elif mode == "slerp":
        hybrid = slerp(alpha, t_flat, matched)
    else:
        raise ValueError(f"Unknown mode: '{mode}'. Use 'lerp' or 'slerp'.")

    # (T_target, D) → (1, D, T_target)
    hybrid_latents = hybrid.T.unsqueeze(0).reshape(orig_shape)

    return hybrid_latents, match_indices

In [56]:
latent_mixed, _ = tokui_style_transfer(latent1, latent2, alpha=0.6, mode="slerp")

In [57]:
from IPython.display import display, Audio
wv_rec = encdec.decode(latent_mixed)
display(Audio(wv_rec, rate=sr))